# Moon Chat v0.1 — Reproducible Google Colab NVIDIA T4 Training Pipeline

**Ownership Policy:** `STRICT_SOVEREIGN_RANDOM_INIT_ONLY`  
**Model Architecture:** Moon Chat 25.7M (`25,698,816` parameters: 8 layers, 8 heads, 512 dim, `block_size=512`, `vocab_size=512`, pure Gaussian random init `N(0, 0.02^2)`).  
**Strict Guarantees:** Zero third-party base weights, zero AI-teacher outputs, zero deterministic template data.

This notebook **refuses to run production training** unless ALL fail-closed security and data-truth checks pass:
1. Physical NVIDIA T4 GPU & PyTorch/CUDA verification (`scripts/validate_t4_empirical.check_t4_hardware_identity`).
2. Pinned Git commit SHA and clean repository verification.
3. Exact SHA-256 verification of dataset manifests, tokenizer, and `training/chat/v0_1_t4_release_config.yaml`.
4. Production Dataset Truth Gate (`DATASET_READY`, zero `DETERMINISTIC_TEMPLATE`/`SYNTHETIC_FIXTURE`/`AI_GENERATED`/`UNVERIFIED`, zero cross-split hash/prompt/reply leakage).
5. Owner Authorization Lock (`core.policy.AUTHORIZED_BY_PREM == True`).


In [ ]:
# Cell 1: Repository Setup, Optional Google Drive Persistence & Commit SHA Verification
import os
import subprocess
import sys

REPO_DIR = os.environ.get("MOON_AI_REPO_DIR", "/content/NanoGPT")
EXPECTED_BRANCH = "feature/phase-2-7-5-training-lock-audit-closure"
# Optionally pin an exact commit SHA prior to launch (leave empty only for inspection dry-runs)
EXPECTED_COMMIT_SHA = os.environ.get("MOON_AI_EXPECTED_COMMIT_SHA", "").strip()
USE_DRIVE_PERSISTENCE = False
DRIVE_CKPT_DIR = "/content/drive/MyDrive/MoonAI/checkpoints/chat_v0_1_release"

if USE_DRIVE_PERSISTENCE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
        print(f"Drive persistence enabled: {DRIVE_CKPT_DIR}")
    except ImportError:
        print("Not running in Google Colab; skipping Drive mount.")

if os.path.isdir(os.path.join(os.getcwd(), "training", "chat")):
    REPO_DIR = os.getcwd()
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
else:
    raise FileNotFoundError(f"Repository directory not found: {REPO_DIR}. Clone Prem759-0/NanoGPT and checkout {EXPECTED_BRANCH}.")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
git_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
print(f"Repo Dir:    {REPO_DIR}")
print(f"Git Branch:  {git_branch}")
print(f"Git Commit:  {git_sha}")

if EXPECTED_COMMIT_SHA and not git_sha.startswith(EXPECTED_COMMIT_SHA):
    raise RuntimeError(f"Commit SHA mismatch! Expected prefix '{EXPECTED_COMMIT_SHA}', got '{git_sha}'.")


In [ ]:
# Cell 2: Verify NVIDIA T4 GPU Availability, VRAM, and PyTorch/CUDA Readiness
import torch
from scripts.validate_t4_empirical import check_t4_hardware_identity

print(f"PyTorch:        {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for Moon Chat v0.1 T4 training. Enable GPU runtime (T4) in Colab.")

gpu_info = check_t4_hardware_identity()
print(f"GPU Device:     {gpu_info['device_name']}")
print(f"Total VRAM MB:  {gpu_info['vram_mb']}")
if not gpu_info.get("is_t4", False):
    raise RuntimeError(f"Hardware gate failed: expected physical NVIDIA T4, got {gpu_info}")
print("Verified physical NVIDIA T4 runtime.")


In [ ]:
# Cell 3: Compute & Verify Cryptographic Hashes of Config, Tokenizer, and Production Manifests
import json
import os
from training.chat.config_validator import compute_file_sha256, load_training_config

CONFIG_PATH = "training/chat/v0_1_t4_release_config.yaml"
cfg = load_training_config(CONFIG_PATH)
paths_cfg = cfg["paths"]

artifact_paths = {
    "config": CONFIG_PATH,
    "tokenizer": paths_cfg["tokenizer_path"],
    "train_manifest": paths_cfg["train_manifest"],
    "val_manifest": paths_cfg["val_manifest"],
    "test_manifest": paths_cfg["test_manifest"],
    "provenance_manifest": paths_cfg.get("provenance_manifest", ""),
}

hashes = {}
for label, rel_path in artifact_paths.items():
    if not rel_path:
        continue
    sha = compute_file_sha256(rel_path)
    if not sha:
        raise FileNotFoundError(f"Required file missing for '{label}': {rel_path}")
    hashes[label] = sha
    print(f"{label:20s}: {rel_path} | sha256={sha}")

prov_path = artifact_paths.get("provenance_manifest")
if prov_path and os.path.isfile(prov_path):
    with open(prov_path, "r", encoding="utf-8") as f:
        prov = json.load(f)
    for split_key, hash_key in [("train", "train_manifest"), ("val", "val_manifest"), ("test", "test_manifest")]:
        expected = (prov.get("output_files", {}).get(split_key) or {}).get("sha256")
        if expected and hashes[hash_key] != expected:
            raise ValueError(f"Checksum mismatch for {hash_key}: expected {expected}, got {hashes[hash_key]}")
    print(f"Manifest checksums verified against {prov_path} (dataset_status={prov.get('dataset_status')}).")


In [ ]:
# Cell 4: Preflight Validation, Runtime/Storage Estimates, & Fail-Closed Dry-Run Check
from core.policy import ACTIVE_OWNERSHIP_POLICY, AUTHORIZED_BY_PREM
from training.chat.config_validator import validate_training_config
from training.chat.train import train_moon_chat

validation = validate_training_config(cfg, require_production_release=True, verify_dataset_truth=True)
print(f"Ownership Policy:      {ACTIVE_OWNERSHIP_POLICY}")
print(f"AUTHORIZED_BY_PREM:    {AUTHORIZED_BY_PREM}")
print(f"Valid Release Config:  {validation['valid_config']}")
print(f"Dataset Truth Status:  {validation['dataset_status']}")
print(f"Ready For Training:    {validation['ready_for_training']}")
print(f"Parameters:            {validation['parameters']['total_parameters']:,}")
print(f"Estimated Peak VRAM:   {validation['t4_vram_estimate']['estimated_peak_vram_mb']} MB / {validation['t4_vram_estimate']['t4_total_vram_mb']} MB")
print(f"Tokens/Step & Budget:  {validation['tokens_per_optimizer_step']:,} tok/step | {validation['total_training_token_budget']:,} total tok budget (~15-22 min on T4)")
print(f"Estimated Ckpt Size:   ~350 MB per full checkpoint payload (model + optimizer + config metadata)")

if validation["dataset_blockers"]:
    print("Dataset Blockers:")
    for b in validation["dataset_blockers"]:
        print(f"  - {b}")

dry_res = train_moon_chat(config_path=CONFIG_PATH, dry_run=True, device_override="cuda")
print(f"Dry-Run Preflight Result: {dry_res['status']} (forward={dry_res['forward_executed']}, backward={dry_res['backward_executed']}, ckpt_written={dry_res['checkpoint_written']})")


---
## Owner-Controlled Launch Cell (Intentionally Locked in Phase 2.7.7)

Running the cell below executes production training **only if**:
- `core.policy.AUTHORIZED_BY_PREM` is `True`, **AND**
- `verify_production_manifest_group()` certifies `DATASET_READY` with zero synthetic/template/AI-teacher contamination and zero cross-split leakage, **AND**
- Physical NVIDIA T4 hardware and `v0_1_t4_release_config.yaml` validation pass.

Otherwise, it **raises immediately and performs zero gradient updates or checkpoint writes**.

In [ ]:
# Cell 5: Owner-Controlled Production Launch Cell (Blocked by default)
import shutil
from core.policy import AUTHORIZED_BY_PREM
from training.chat.config_validator import load_training_config, validate_training_config
from training.chat.train import train_moon_chat

if not AUTHORIZED_BY_PREM:
    raise PermissionError(
        "TRAINING BLOCKED: core.policy.AUTHORIZED_BY_PREM is False. "
        "Production training requires explicit owner authorization from Prem Gaikwad."
    )

launch_cfg = load_training_config("training/chat/v0_1_t4_release_config.yaml")
launch_check = validate_training_config(launch_cfg, require_production_release=True, verify_dataset_truth=True)
if not launch_check["ready_for_training"]:
    raise RuntimeError(
        f"TRAINING BLOCKED: status={launch_check['dataset_status']} | blockers={launch_check['blockers']}"
    )

result = train_moon_chat(
    config_path="training/chat/v0_1_t4_release_config.yaml",
    device_override="cuda",
    dry_run=False,
    smoke_test=False,
)
print("Training completed:", result.get("status"), "checkpoint:", result.get("checkpoint_path"))

if USE_DRIVE_PERSISTENCE and result.get("checkpoint_path") and os.path.isfile(result["checkpoint_path"]):
    shutil.copy2(result["checkpoint_path"], DRIVE_CKPT_DIR)
    print(f"Copied checkpoint artifact to Drive: {DRIVE_CKPT_DIR}")
